In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("KafkaToDB") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate() 

spark.sparkContext.setLogLevel("WARN")


In [8]:
df_raw = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "github-repos") \
    .option("startingOffsets", "earliest") \
    .option("endingOffsets", "latest") \
    .load()




In [9]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

metadata_schema = StructType([
    StructField("name", StringType()),
    StructField("owner", StringType()),
    StructField("created_at", StringType()),
    StructField("updated_at", StringType()),
    StructField("language", StringType()),
    StructField("stars", IntegerType())
])

root_schema = StructType([
    StructField("metadata", metadata_schema),
    StructField("collected_at", StringType())
])

df_value = df_raw.selectExpr("CAST(value AS STRING) as raw_value")

df_parsed = df_value.select(from_json(col("raw_value"), root_schema).alias("data"))

df_flat = df_parsed.select(
    col("data.metadata.name").alias("repo_name"),
    col("data.metadata.owner").alias("repo_owner"),
    col("data.metadata.language").alias("language"),
    col("data.metadata.stars").alias("stars"),
    col("data.collected_at").alias("collected_at")
)

# Now show the results once
df_flat.show(truncate=False)


25/07/07 11:02:45 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+--------------------+----------+--------+-----+--------------------------+
|repo_name           |repo_owner|language|stars|collected_at              |
+--------------------+----------+--------+-----+--------------------------+
|Base-de-connaissance|Git-Know  |NULL    |0    |2025-07-07T10:52:54.090764|
|Base-de-connaissance|Git-Know  |NULL    |0    |2025-07-07T10:53:23.562294|
|metasfresh          |Git-Know  |NULL    |0    |2025-07-07T10:54:30.598288|
|evershop            |Git-Know  |NULL    |0    |2025-07-07T10:55:33.405495|
|maybe               |Git-Know  |NULL    |0    |2025-07-07T10:56:36.178387|
|shop-e-commerce     |Git-Know  |NULL    |0    |2025-07-07T10:57:34.354184|
+--------------------+----------+--------+-----+--------------------------+



In [11]:
from pyspark.sql.types import *

commit_schema = StructType([
    StructField("sha", StringType()),
    StructField("author", StringType()),
    StructField("date", StringType()),
    StructField("message", StringType())
])

root_schema = StructType([
    StructField("metadata", metadata_schema),
    StructField("collected_at", StringType()),
    StructField("commits", ArrayType(commit_schema))
])
from pyspark.sql.functions import from_json, col, explode

# Parse raw Kafka string to JSON structure
df_parsed = df_value.select(from_json(col("raw_value"), root_schema).alias("data"))

# Extract metadata and commit info
df_commits = df_parsed \
    .withColumn("repo_name", col("data.metadata.name")) \
    .withColumn("repo_owner", col("data.metadata.owner")) \
    .withColumn("collected_at", col("data.collected_at")) \
    .withColumn("commit", explode(col("data.commits"))) \
    .select(
        col("repo_name"),
        col("repo_owner"),
        col("collected_at"),
        col("commit.sha").alias("commit_sha"),
        col("commit.author").alias("commit_author"),
        col("commit.date").alias("commit_date"),
        col("commit.message").alias("commit_message")
    )
df_commits.show()

25/07/07 11:32:17 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+
|           repo_name|repo_owner|        collected_at|          commit_sha|       commit_author|         commit_date|      commit_message|
+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+
|Base-de-connaissance|  Git-Know|2025-07-07T10:52:...|b3e421445b632ccc8...|             unknown|2025-07-02T17:11:47Z|  add docker compose|
|Base-de-connaissance|  Git-Know|2025-07-07T10:53:...|b3e421445b632ccc8...|             unknown|2025-07-02T17:11:47Z|  add docker compose|
|          metasfresh|  Git-Know|2025-07-07T10:54:...|aaedd46717e846b6b...|           Max Rieck|2025-06-26T09:40:11Z|fix .mergify.yml ...|
|          metasfresh|  Git-Know|2025-07-07T10:54:...|6188464b9e27618f0...|metasfresh-onboar...|2025-06-16T09:16:04Z|Added temporary_t...|
|          metasfresh|  Git